# SPECTER2 CORAL Metadata CV5 Ensemble

This notebook trains a SPECTER2-based CORAL ordinal regression model for academic paper label prediction.

Main features:
1. SPECTER2 encoder: `allenai/specter2_base`
2. Rich metadata text: title, venue, year, authors, DOI
3. Venue embedding
4. CORAL ordinal regression head
5. QWK-based checkpoint selection
6. 5-fold CV ensemble
7. Colab + GitHub setup
8. Final output saved to `submissions/submission.csv`

Experiment name: `specter2_coral_metadata_cv5_ensemble`


## Cell 0 — Colab GitHub Setup

Run this cell on Google Colab. Replace `GITHUB_REPO` with your real GitHub repository URL.


In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"

REPO_NAME = Path(GITHUB_REPO).stem.replace(".git", "")
PROJECT_ROOT = Path("/content") / REPO_NAME

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    print("Running on Google Colab.")

    if "YOUR_USERNAME" in GITHUB_REPO or "YOUR_REPO" in GITHUB_REPO:
        raise ValueError("Please replace GITHUB_REPO with your real GitHub repository URL.")

    if not PROJECT_ROOT.exists():
        print("Cloning:", GITHUB_REPO)
        subprocess.run(["git", "clone", GITHUB_REPO, str(PROJECT_ROOT)], check=True)
    else:
        print("Repository already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

    os.chdir(PROJECT_ROOT)
    print("Current working directory:", Path.cwd())

    for file_name in ["train.csv", "public_test.csv", "private_test.csv"]:
        p = PROJECT_ROOT / "data" / file_name
        if not p.exists():
            raise FileNotFoundError(f"Missing required file: {p}")

    print("All required data files found.")
else:
    print("Not running on Colab. Skipping GitHub clone.")
    print("Current working directory:", Path.cwd())


## Cell 0.1 — Install Required Packages on Colab

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q numpy pandas scipy scikit-learn torch transformers tqdm
else:
    print("Not running on Colab. Make sure your local environment has the required packages.")


## Cell 1 — Imports and Configuration

In [ ]:
'''
1.
MAX_LENGTH = 192
BACKBONE_LR = 1e-5
UNFREEZE_LAST_N_LAYERS = 3
EPOCHS = 8

2.
MAX_LENGTH = 192
BACKBONE_LR = 1e-5
UNFREEZE_LAST_N_LAYERS = 2
EPOCHS = 8

3.
MAX_LENGTH = 192
BACKBONE_LR = 1e-5
SEED = 52
EPOCHS = 8

4.
MAX_LENGTH = 192
BACKBONE_LR = 1e-5
SEED = 62
EPOCHS = 8
'''

import os
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 5

MODEL_NAME = "allenai/specter2_base"

MAX_LENGTH = 160
BATCH_SIZE = 8
EPOCHS = 8

BACKBONE_LR = 1e-5 # [1e-5, 7e-6]
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01

VENUE_DIM = 32
DROPOUT = 0.25
N_DROPOUT_SAMPLES = 4
UNFREEZE_LAST_N_LAYERS = 3 # [2, 3]

LABEL_COL = "Label"
EXPERIMENT_NAME = "specter2_coral_metadata_cv5_ensemble"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Model:", MODEL_NAME)
print("Experiment:", EXPERIMENT_NAME)


## Cell 2 — Load Data

The notebook expects `data/train.csv`, `data/public_test.csv`, and `data/private_test.csv`.


In [ ]:
def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent, Path("/content")]

    content_root = Path("/content")
    if content_root.exists():
        for child in content_root.iterdir():
            if child.is_dir():
                candidates.append(child)

    seen = set()
    unique_candidates = []
    for p in candidates:
        p = p.resolve()
        if p not in seen:
            seen.add(p)
            unique_candidates.append(p)

    for p in unique_candidates:
        if (p / "data" / "train.csv").exists():
            return p

    raise FileNotFoundError(
        "Could not find data/train.csv. On Colab, run the GitHub setup cell first."
    )

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models" / EXPERIMENT_NAME
SUBMISSION_DIR = ROOT / "submissions"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
public_test = pd.read_csv(DATA_DIR / "public_test.csv")
private_test = pd.read_csv(DATA_DIR / "private_test.csv")

if LABEL_COL not in train.columns and "label" in train.columns:
    LABEL_COL = "label"

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)
print("Submission directory:", SUBMISSION_DIR)
print("Train:", train.shape)
print("Public:", public_test.shape)
print("Private:", private_test.shape)
print("Label distribution:")
print(train[LABEL_COL].value_counts().sort_index())

print("Train hash:")
print(pd.util.hash_pandas_object(
    train[["id", "title", "venue", "year", "authors", "doi", LABEL_COL]],
    index=True
).sum())


## Cell 3 — Build Rich Text and Venue IDs

SPECTER2 receives a structured text representation of each paper. Venue is also used as a separate embedding feature.


In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def build_input_text(df):
    return (
        "Title: " + df["title"].map(clean_text) +
        " [SEP] Venue: " + df["venue"].map(clean_text) +
        " [SEP] Year: " + df["year"].map(clean_text) +
        " [SEP] Authors: " + df["authors"].map(clean_text) +
        " [SEP] DOI: " + df["doi"].map(clean_text)
    )

train["input_text"] = build_input_text(train)
public_test["input_text"] = build_input_text(public_test)
private_test["input_text"] = build_input_text(private_test)

venue2id = {"<UNK>": 0}
for v in sorted(train["venue"].fillna("UNKNOWN").astype(str).str.lower().unique()):
    if v not in venue2id:
        venue2id[v] = len(venue2id)

def map_venue(x):
    if pd.isna(x):
        return 0
    return venue2id.get(str(x).lower(), 0)

train["venue_id"] = train["venue"].map(map_venue)
public_test["venue_id"] = public_test["venue"].map(map_venue)
private_test["venue_id"] = private_test["venue"].map(map_venue)

NUM_VENUES = len(venue2id)

print("Number of venues:", NUM_VENUES)
train[["input_text", "venue_id", LABEL_COL]].head()


## Cell 4 — QWK and CORAL Utilities

CORAL predicts four rank conditions: `label > 1`, `label > 2`, `label > 3`, and `label > 4`.


In [ ]:
def quadratic_weighted_kappa(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def labels_to_coral_targets(labels, num_classes=5):
    labels = np.asarray(labels).astype(int)
    targets = np.zeros((len(labels), num_classes - 1), dtype=np.float32)
    for i, label in enumerate(labels):
        targets[i, :label - 1] = 1.0
    return targets

def coral_logits_to_continuous(logits):
    probs = torch.sigmoid(logits)
    return 1.0 + probs.sum(dim=1)

def apply_thresholds(pred, thresholds):
    return np.digitize(np.asarray(pred), thresholds) + 1

def optimize_thresholds(y_true, pred, initial=None):
    if initial is None:
        initial = np.array([1.5, 2.5, 3.5, 4.5], dtype=float)

    def loss(thresholds):
        thresholds = np.sort(thresholds)
        y_hat = apply_thresholds(pred, thresholds)
        return -quadratic_weighted_kappa(y_true, y_hat)

    result = minimize(
        loss,
        x0=initial,
        method="Nelder-Mead",
        options={"maxiter": 2000, "xatol": 1e-6, "fatol": 1e-6},
    )

    thresholds = np.sort(result.x)
    score = -result.fun
    return thresholds, score


## Cell 5 — Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PaperDataset(Dataset):
    def __init__(self, df, labels=None, max_length=160):
        self.texts = df["input_text"].values
        self.venue_ids = df["venue_id"].values.astype(np.int64)
        self.labels = labels
        self.max_length = max_length
        self.coral_targets = labels_to_coral_targets(labels) if labels is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "venue_id": torch.tensor(self.venue_ids[idx], dtype=torch.long),
        }

        if self.labels is not None:
            item["coral_target"] = torch.tensor(self.coral_targets[idx], dtype=torch.float)
            item["label"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


## Cell 6 — SPECTER2 CORAL Model

In [ ]:
class CoralLayer(nn.Module):
    def __init__(self, input_dim, num_classes=5):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1, bias=False)
        self.bias = nn.Parameter(torch.zeros(num_classes - 1))

    def forward(self, x):
        return self.linear(x) + self.bias


class Specter2CoralModel(nn.Module):
    def __init__(
        self,
        model_name,
        num_venues,
        num_classes=5,
        venue_dim=32,
        dropout=0.25,
        n_dropout_samples=4,
        unfreeze_last_n_layers=3,
    ):
        super().__init__()

        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size

        for param in self.backbone.parameters():
            param.requires_grad = False

        if hasattr(self.backbone, "encoder") and hasattr(self.backbone.encoder, "layer"):
            total_layers = len(self.backbone.encoder.layer)
            for layer_idx in range(total_layers - unfreeze_last_n_layers, total_layers):
                for param in self.backbone.encoder.layer[layer_idx].parameters():
                    param.requires_grad = True

        if hasattr(self.backbone, "pooler") and self.backbone.pooler is not None:
            for param in self.backbone.pooler.parameters():
                param.requires_grad = True

        self.venue_embedding = nn.Embedding(num_venues, venue_dim)
        feature_dim = hidden_size * 2 + venue_dim

        self.feature_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.ReLU(),
            nn.LayerNorm(256),
            nn.Dropout(dropout),
        )

        self.dropouts = nn.ModuleList([
            nn.Dropout(dropout) for _ in range(n_dropout_samples)
        ])

        self.coral = CoralLayer(256, num_classes=num_classes)

    def mean_max_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        mean_pool = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        masked_hidden = last_hidden_state.masked_fill(mask == 0, -1e9)
        max_pool = masked_hidden.max(dim=1).values
        return torch.cat([mean_pool, max_pool], dim=1)

    def forward(self, input_ids, attention_mask, venue_id):
        output = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_max_pooling(output.last_hidden_state, attention_mask)
        venue_emb = self.venue_embedding(venue_id)
        features = torch.cat([pooled, venue_emb], dim=1)
        hidden = self.feature_head(features)

        logits = 0
        for dropout in self.dropouts:
            logits = logits + self.coral(dropout(hidden))

        logits = logits / len(self.dropouts)
        return logits


## Cell 7 — Training Utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, leave=False):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        venue_id = batch["venue_id"].to(device)
        coral_target = batch["coral_target"].to(device)

        logits = model(input_ids, attention_mask, venue_id)
        loss = criterion(logits, coral_target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * input_ids.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_continuous(model, loader):
    model.eval()
    preds = []

    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        venue_id = batch["venue_id"].to(device)

        logits = model(input_ids, attention_mask, venue_id)
        continuous = coral_logits_to_continuous(logits)
        preds.append(continuous.cpu().numpy())

    return np.concatenate(preds)


def build_optimizer(model):
    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("backbone"):
            backbone_params.append(param)
        else:
            head_params.append(param)

    return torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": BACKBONE_LR},
            {"params": head_params, "lr": HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )


## Cell 8 — 5-Fold Cross-Validation Training

The best model for each fold is selected by validation QWK, not by loss.


In [ ]:
y = train[LABEL_COL].astype(int).values

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_pred = np.zeros(len(train), dtype=np.float32)
fold_thresholds = []
fold_scores = []
model_paths = []

print("=" * 60)
print("Training configuration")
print("=" * 60)
print("Method: SPECTER2 + CORAL + Metadata")
print("N_SPLITS:", N_SPLITS)
print("EPOCHS:", EPOCHS)
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("UNFREEZE_LAST_N_LAYERS:", UNFREEZE_LAST_N_LAYERS)
print("BACKBONE_LR:", BACKBONE_LR)
print("HEAD_LR:", HEAD_LR)
print("=" * 60)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y), 1):
    print(f"\n========== Fold {fold}/{N_SPLITS} ==========")

    seed_everything(SEED + fold)

    train_df = train.iloc[tr_idx].reset_index(drop=True)
    valid_df = train.iloc[va_idx].reset_index(drop=True)

    y_train = y[tr_idx]
    y_valid = y[va_idx]

    train_dataset = PaperDataset(train_df, labels=y_train, max_length=MAX_LENGTH)
    valid_dataset = PaperDataset(valid_df, labels=y_valid, max_length=MAX_LENGTH)

    generator = torch.Generator()
    generator.manual_seed(SEED + fold)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model = Specter2CoralModel(
        model_name=MODEL_NAME,
        num_venues=NUM_VENUES,
        num_classes=5,
        venue_dim=VENUE_DIM,
        dropout=DROPOUT,
        n_dropout_samples=N_DROPOUT_SAMPLES,
        unfreeze_last_n_layers=UNFREEZE_LAST_N_LAYERS,
    ).to(device)

    optimizer = build_optimizer(model)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * 0.1)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    criterion = nn.BCEWithLogitsLoss()

    best_qwk = -1.0
    best_thresholds = None
    best_model_path = MODEL_DIR / f"specter2_coral_fold_{fold}.pt"

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion)
        valid_pred = predict_continuous(model, valid_loader)
        thresholds, valid_qwk = optimize_thresholds(y_valid, valid_pred)

        print(
            f"Fold {fold} | Epoch {epoch} | "
            f"Loss: {train_loss:.5f} | "
            f"QWK: {valid_qwk:.6f} | "
            f"Thresholds: {thresholds}"
        )

        if valid_qwk > best_qwk:
            best_qwk = valid_qwk
            best_thresholds = thresholds
            torch.save(model.state_dict(), best_model_path)

    print(f"Best Fold {fold} QWK: {best_qwk:.6f}")
    print(f"Best Fold {fold} thresholds: {best_thresholds}")

    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.to(device)

    best_valid_pred = predict_continuous(model, valid_loader)
    oof_pred[va_idx] = best_valid_pred

    fold_thresholds.append(best_thresholds)
    fold_scores.append(best_qwk)
    model_paths.append(best_model_path)

    del model
    torch.cuda.empty_cache()

print("\n========== Cross-Validation Summary ==========")
print("Fold QWK scores:", fold_scores)
print("Mean fold QWK:", np.mean(fold_scores))
print("Std fold QWK:", np.std(fold_scores))

oof_thresholds, oof_qwk = optimize_thresholds(y, oof_pred)
oof_labels = apply_thresholds(oof_pred, oof_thresholds)

print("\nOOF Optimized QWK:", oof_qwk)
print("OOF Thresholds:", oof_thresholds)
print("OOF Prediction Distribution:")
print(pd.Series(oof_labels).value_counts().sort_index())


## Cell 9 — Save OOF Predictions

In [ ]:
oof_df = pd.DataFrame({
    "id": train["id"].values,
    "y_true": y,
    "pred_continuous": oof_pred,
    "pred_label": apply_thresholds(oof_pred, oof_thresholds),
})

oof_path = SUBMISSION_DIR / "oof_specter2_coral_metadata_cv5.csv"
oof_df.to_csv(oof_path, index=False)

print("Saved OOF predictions to:", oof_path)
oof_df.head()


## Cell 10 — Inference and Submission

The notebook saves:

```text
submissions/submission.csv
submissions/submission_specter2_coral_metadata_cv5.csv
```


In [ ]:
public_dataset = PaperDataset(public_test, labels=None, max_length=MAX_LENGTH)
private_dataset = PaperDataset(private_test, labels=None, max_length=MAX_LENGTH)

public_loader = DataLoader(
    public_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

private_loader = DataLoader(
    private_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

public_pred = np.zeros(len(public_test), dtype=np.float32)
private_pred = np.zeros(len(private_test), dtype=np.float32)

for fold, model_path in enumerate(model_paths, 1):
    print(f"Inference with fold {fold}: {model_path}")

    model = Specter2CoralModel(
        model_name=MODEL_NAME,
        num_venues=NUM_VENUES,
        num_classes=5,
        venue_dim=VENUE_DIM,
        dropout=DROPOUT,
        n_dropout_samples=N_DROPOUT_SAMPLES,
        unfreeze_last_n_layers=UNFREEZE_LAST_N_LAYERS,
    ).to(device)

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)

    public_pred += predict_continuous(model, public_loader) / len(model_paths)
    private_pred += predict_continuous(model, private_loader) / len(model_paths)

    del model
    torch.cuda.empty_cache()

public_labels = apply_thresholds(public_pred, oof_thresholds)
private_labels = apply_thresholds(private_pred, oof_thresholds)

submission = pd.concat(
    [
        pd.DataFrame({"id": public_test["id"].values, "Label": public_labels.astype(int)}),
        pd.DataFrame({"id": private_test["id"].values, "Label": private_labels.astype(int)}),
    ],
    axis=0,
    ignore_index=True,
)

submission_path = SUBMISSION_DIR / "submission.csv"
submission_exp_path = SUBMISSION_DIR / "submission_specter2_coral_metadata_cv5.csv"

submission.to_csv(submission_path, index=False)
submission.to_csv(submission_exp_path, index=False)

print("Saved submission to:", submission_path)
print("Saved experiment copy to:", submission_exp_path)
print("Submission shape:", submission.shape)
print("Submission label distribution:")
print(submission["Label"].value_counts().sort_index())

submission.head()


## Cell 11 — Next Experiments

If this SPECTER2 + CORAL model is promising, try:

1. `UNFREEZE_LAST_N_LAYERS = 2` versus `3`.
2. `EPOCHS = 5`, `8`, or `10`.
3. Frozen SPECTER2 embeddings + Ridge/LightGBM.
4. Add very light contrastive loss: `0.001–0.005`.
5. Ensemble SPECTER2-CORAL with the previous best SciBERT contrastive regression model.
